<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/fabiobento/lab-cont-2026-2/blob/main/modulo1_introducao/labs/lab04_resposta_2a_ordem_e_projeto_P.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
  </td>
</table>

# Lab 04 — Resposta ao Degrau de 2ª Ordem e Projeto de Controle P

**Módulo 01 · Semana 5 · Laboratório de Controle Automático (Ifes Guarapari)**

Neste laboratório você vai:
1. Medir Mp, tp, tr, ts(5%) numa curva simulada e confrontar com as fórmulas;
2. Ver o efeito de ζ, ωn, σ e ωd na resposta;
3. Fazer os **dois projetos do curso**: k para Mp ≤ 10% e k para tp = 3,14 s;
4. Entregar o relatório (esta é a **nota do Projeto P**).

**Teoria de apoio:** `teoria_modulo1.md`, §1.4 (fórmulas e projetos).

In [ ]:
# %% IMPORTS (rode esta célula primeiro)
!pip install --quiet control==0.10.2
import control as ct
import numpy as np
import matplotlib.pyplot as plt
s = ct.tf('s')
plt.rcParams['figure.figsize'] = (9, 4)
plt.rcParams['axes.grid'] = True
print('python-control', ct.__version__)

## Parte 1 — Medindo as quatro grandezas

Funções de medição (use em TODO o curso). Depois aplicamos ao exemplo canônico
G(s) = 25/(s² + 4s + 25): as fórmulas (§1.4.5) dão Mp = 25,4%, tp = 0,69 s,
tr = 0,43 s, ts(5%) ≈ 1,5 s. Bate?

In [ ]:
def medir(T, t=None):
    '''Mede Mp(%), tp, tr(0-100%), ts(±5%) da resposta ao degrau unitário de T.'''
    if t is None: t = np.linspace(0, 50, 5000)
    t, y = ct.step_response(T, t)
    t, y = np.asarray(t), np.asarray(y)
    yss = y[-1]
    Mp = 100*(y.max() - yss)/yss
    tp = t[np.argmax(y)]
    ic = np.argmax(y >= yss)          # 1º cruzamento do valor final
    tr = t[ic]
    fora = np.where(np.abs(y - yss) > 0.05*abs(yss))[0]
    ts = t[fora[-1]] if len(fora) else 0.0
    return dict(Mp=Mp, tp=tp, tr=tr, ts=ts, yss=yss, t=t, y=y)

G = 25/(s**2 + 4*s + 25)
m = medir(G, np.linspace(0, 4, 4000))
print(f"medido:   Mp = {m['Mp']:.1f}%  tp = {m['tp']:.3f}s  tr = {m['tr']:.3f}s  ts = {m['ts']:.3f}s")
print('teórico:  Mp = 25.4%  tp = 0.686s  tr = 0.433s  ts = 1.500s (3/σ, pessimista)')
plt.plot(m['t'], m['y'])
plt.axhline(m['yss'], color='k', ls='--', alpha=.4)
plt.axhline(1.05*m['yss'], color='r', ls=':', alpha=.5); plt.axhline(0.95*m['yss'], color='r', ls=':', alpha=.5)
plt.title('G(s) = 25/(s² + 4s + 25)'); plt.xlabel('t (s)'); plt.ylabel('y(t)'); plt.show()

## Parte 2 — A geometria dos polos na resposta

Varie UM parâmetro por vez na FT padrão ωn²/(s² + 2ζωn·s + ωn²) e observe
(§1.4.4 — compare com a figura m1_fig18 da teoria).

In [ ]:
t = np.linspace(0, 8, 800)

# (a) varia ζ com ωn fixo -> muda Mp
for zeta in [0.2, 0.4, 0.7, 1.0]:
    wn = 2.0
    T = wn**2/(s**2 + 2*zeta*wn*s + wn**2)
    tt, y = ct.step_response(T, t)
    plt.plot(tt, y, label=f'ζ={zeta}')
plt.title('(a) ωn = 2 fixo, variando ζ'); plt.legend(); plt.show()

# (b) varia ωn com ζ fixo -> muda velocidade, Mp igual
for wn in [1.0, 2.0, 4.0]:
    zeta = 0.4
    T = wn**2/(s**2 + 2*zeta*wn*s + wn**2)
    m = medir(T, t)
    plt.plot(m['t'], m['y'], label=f"ωn={wn}  (Mp={m['Mp']:.1f}%)")
plt.title('(b) ζ = 0,4 fixo, variando ωn — Mp não muda!'); plt.legend(); plt.show()

## Parte 3 — PROJETO 1: k para Mp ≤ 10%   (entregar no relatório)

Planta G(s) = 1/[s(s+1)], controle P, MF unitária → T(s) = k/(s² + s + k).
Passos (§1.4.6): ζ de Mp → ωn = 1/(2ζ) → k = ωn². Depois **verifique na simulação**.

In [ ]:
import math
Mp_esp = 0.10
zeta = math.sqrt(math.log(Mp_esp)**2/(math.pi**2 + math.log(Mp_esp)**2))
wn = 1/(2*zeta)
k = wn**2
print(f'ζ = {zeta:.4f}   ωn = {wn:.4f}   k = {k:.3f}')

G = 1/(s*(s+1))
T = ct.feedback(k*G)
m = medir(T, np.linspace(0, 15, 1500))
print(f"verificação: Mp medido = {m['Mp']:.1f}% (especificado: ≤ 10%)")
plt.plot(m['t'], m['y'], label=f'k = {k:.3f}')
plt.axhline(1.10, color='r', ls='--', label='limite Mp = 10%')
plt.legend(); plt.title('Projeto 1 — Mp ≤ 10%'); plt.xlabel('t (s)'); plt.ylabel('y(t)'); plt.show()

## Parte 4 — PROJETO 2: k para tp = 3,14 s   (entregar no relatório)

Mesma planta. Passos (§1.4.6): ωd = π/tp → σ = 0,5 (fixo pela planta!) →
k = σ² + ωd². Verifique e **meça também o Mp que saiu de brinde**.

In [ ]:
tp_esp = 3.14
wd = math.pi/tp_esp
sigma = 0.5                       # 2ζωn = 1 -> σ = ζωn = 0,5 (a planta manda)
k = sigma**2 + wd**2
print(f'ωd = {wd:.4f}   k = {k:.3f}   polos = {np.round(ct.poles(ct.feedback(k*G)),3)}')

T = ct.feedback(k*G)
m = medir(T, np.linspace(0, 20, 2000))
print(f"verificação: tp medido = {m['tp']:.3f}s (especificado: 3,14s)   Mp = {m['Mp']:.1f}%")
plt.plot(m['t'], m['y'], label=f'k = {k:.3f}')
plt.legend(); plt.title('Projeto 2 — tp = 3,14 s'); plt.xlabel('t (s)'); plt.ylabel('y(t)'); plt.show()

## Parte 5 — Relatório (Projeto P — nota)

Entregue um PDF com:
1. **Projeto 1**: dedução de k no papel (foto ou digitada), valor de k, gráfico da
   simulação com Mp medido, e comentário: o Mp medido atendeu? Por que não é exatamente 10%?
2. **Projeto 2**: idem para tp, incluindo os polos resultantes e o Mp medido.
3. **Discussão**: tente agora atender Mp ≤ 10% E tp = 3,14 s **simultaneamente** com
   um único k. Mostre numericamente que é impossível (dica: cada especificação fixa
   um lugar diferente para os polos — §1.4.4) e conclua: por que precisaremos de
   controladores mais ricos que o P?
4. **Extra (vale bônus)**: repita o Projeto 1 para a planta 1/[s(s+4)] usando a
   fórmula literal k = a²/(4ζ²) — confira com o Exercício 1.4.5 dos exercícios resolvidos
   (k = 11,45).